# MODE 2 — M2_F03 SCENOGRAPHY DOCK — CONTROL

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

**Rôle :** Diagnostics pré-vol M2_F03.

| Loi | Règle |
|-----|-------|
| R-01 | Isolation Mode 2 |
| R-05 | GLB décor fourni par Opérateur |

In [ ]:
#@title 🔗 [EXODUS] Drive + Session JSON
#@markdown Monte le Drive et lit exodus_session.json genere par EXO_LAUNCHER
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/EXODUS_V2"  #@param {type:"string"}
sys.path.insert(0, DRIVE_ROOT)

_session_path = Path(DRIVE_ROOT) / "exodus_session.json"
if _session_path.exists():
    with open(_session_path) as _f:
        EXODUS_SESSION = json.load(_f)
    print("OK exodus_session.json charge")
    print(f"  Mode     : {EXODUS_SESSION['mode']} --- {EXODUS_SESSION['mode_label']}")
    print(f"  Timestamp: {EXODUS_SESSION['timestamp']}")
    print(f"  Drive    : {EXODUS_SESSION['drive_root']}")
else:
    print("ATTENTION : exodus_session.json introuvable")
    print("   -> Lancer EXO_LAUNCHER.ipynb d'abord.")
    EXODUS_SESSION = {
        "mode": None, "mode_label": "UNKNOWN",
        "drive_root": DRIVE_ROOT, "status": "missing"
    }

In [ ]:
from pathlib import Path
import json

FREGATE_ROOT = Path("../")
dirs = {
    "IN_GLB_DECOR":  FREGATE_ROOT / "IN_GLB_DECOR",
    "IN_GLB_AVATAR": FREGATE_ROOT / "IN_GLB_AVATAR",
    "IN_AUDIO":      FREGATE_ROOT / "IN_AUDIO",
    "OUT_SCENE":     FREGATE_ROOT / "OUT_SCENE",
    "OUT_REPORT":    FREGATE_ROOT / "OUT_REPORT",
}
print("=== M2_F03 CONTROL — PRÉ-VOL ===")
for name, d in dirs.items():
    status = "✅" if d.exists() else "❌"
    print(f"  {status} {name}")

In [ ]:
# Inspect GLB inputs
print("=== GLB INPUTS ===")
for label, d in [("DÉCOR", dirs["IN_GLB_DECOR"]), ("AVATAR", dirs["IN_GLB_AVATAR"])]:
    glbs = list(d.glob("*.glb")) if d.exists() else []
    print(f"  {label} : {len(glbs)} fichier(s)")
    for g in glbs:
        size_mb = g.stat().st_size / (1024*1024)
        with open(g, "rb") as f:
            magic = f.read(4)
        icon = "✅" if magic == b"glTF" else "❌"
        print(f"    {icon} {g.name} ({size_mb:.2f} MB)")

In [ ]:
# Check Blender
import subprocess, shutil
blender_path = shutil.which("blender")
if blender_path:
    result = subprocess.run([blender_path, "--version"], capture_output=True, text=True)
    print(f"✅ Blender : {blender_path}")
    print(f"   {result.stdout.strip().splitlines()[0]}")
else:
    print("⚠️  Blender non trouvé dans PATH — vérifier installation")

In [ ]:
# Lecture rapport si existant
report_path = dirs["OUT_REPORT"] / "m2_f03_report.json"
if report_path.exists():
    with open(report_path) as f:
        r = json.load(f)
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"{icon} Statut : {r['status']} ({r['timestamp']})")
    print(f"   Inputs : {r.get('inputs')}")
    br = r.get("blender_run", {}).get("internal", {})
    if br:
        print(f"   Décor importé   : {br.get('decor_imported')}")
        print(f"   Avatar importé  : {br.get('avatar_imported')}")
        print(f"   Shadow catcher  : {br.get('shadow_catcher')}")
        print(f"   HDRi appliqué   : {br.get('hdri_applied')}")
        print(f"   Objets totaux   : {br.get('object_count')}")
else:
    print("Aucun rapport — lancer la production d'abord")